# 06 — 无机与晶体材料信息学：随堂练习参考答案（Solutions）

本 Notebook 汇总了第六讲 `06a_crystal_structures_and_databases.ipynb` 与 `06b_xrd_phase_diagram_screening.ipynb` 中全部 6 道随堂练习（`06a-1` 至 `06b-3`）的完整可运行参考代码与材料学解析。

---

## 0. 公共环境

In [ ]:
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from scipy.spatial import ConvexHull

warnings.filterwarnings("ignore")

from pymatgen.core import Structure, Lattice, Composition, Element, Species
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.io.cif import CifParser
from pymatgen.analysis.diffraction.xrd import XRDCalculator
from pymatgen.entries.computed_entries import ComputedEntry
from pymatgen.analysis.phase_diagram import PhaseDiagram

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

plt.rcParams["figure.dpi"] = 110
np.random.seed(42)

def find_file(name):
    for path in (f"../data/{name}", f"data/{name}", name):
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"找不到 {name}")

xrd = XRDCalculator(wavelength="CuKa")

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（solutions/ 子目录与 Colab 通用）
import pathlib, sys

for _p in (pathlib.Path.cwd().parent, pathlib.Path.cwd().parent.parent):
    sys.path.insert(0, str(_p))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---

# `06a` — 晶体结构与数据库练习答案

### 练习 06a-1：搭建卤化物钙钛矿并比较晶胞

**题目回顾**：用 `Pm-3m` 搭出 $\text{SrTiO}_3$、$\text{BaTiO}_3$、$\text{CsPbI}_3$，打印晶胞信息与 B–X 键长，验证 $a \approx 2 d_{B-X}$。

In [ ]:
PEROVSKITES = {
    "SrTiO3": (3.905, "Sr", "Ti", "O"),
    "BaTiO3": (4.006, "Ba", "Ti", "O"),
    "CsPbI3": (6.289, "Cs", "Pb", "I"),
}

def cubic_perovskite(a, A, B, X):
    return Structure.from_spacegroup(
        "Pm-3m", Lattice.cubic(a), [A, B, X],
        [[0, 0, 0], [0.5, 0.5, 0.5], [0.5, 0.5, 0.0]])

rows, structs = [], {}
for name, (a, A, B, X) in PEROVSKITES.items():
    s = cubic_perovskite(a, A, B, X)
    structs[name] = s
    # B 位和 X 位的编号要现找，不能写死
    i_B = next(i for i, st in enumerate(s.sites) if st.specie.symbol == B)
    idx_X = [i for i, st in enumerate(s.sites) if st.specie.symbol == X]
    # 周期性边界下必须对所有 X 取最小值
    d_BX = min(s.get_distance(i_B, j) for j in idx_X)
    rows.append({"material": name, "a_A": a, "n_sites": s.num_sites,
                 "V_A3": round(s.volume, 2), "rho_g_cm3": round(s.density, 3),
                 "d_BX_A": round(d_BX, 4), "a_over_2dBX": round(a / (2 * d_BX), 4)})

tab = pd.DataFrame(rows)
print(tab.to_string(index=False))
print(f"\nV(CsPbI3) / V(SrTiO3) = {tab.V_A3.iloc[2] / tab.V_A3.iloc[0]:.2f}")

**解析**：最后一列全部**精确等于 1.000**。这不是巧合，而是立方钙钛矿的几何恒等式：B 位在体心 $(\frac{1}{2},\frac{1}{2},\frac{1}{2})$，最近的 X 位在面心 $(\frac{1}{2},\frac{1}{2},0)$，两者只差 $\frac{1}{2}$ 个 $c$ 轴，所以 $d_{B-X} = a/2$ 严格成立。

$\text{CsPbI}_3$ 的晶胞体积是 $\text{SrTiO}_3$ 的约 **4.2 倍**。这个“大”带来了卤化物钙钛矿一系列独特的物理性质：

- 离子半径大 $\Rightarrow$ 晶格软、声子频率低 $\Rightarrow$ 强电声耦合与**极化子**行为；
- Pb 的 $6s^2$ 孤对电子加上重元素效应 $\Rightarrow$ 介电常数高 $\Rightarrow$ **激子结合能低**（几十 meV），室温下光生载流子可以直接解离，这正是钙钛矿太阳能电池效率高的根本原因之一；
- 对比之下 $\text{SrTiO}_3$ 虽然也是钙钛矿，却是带隙 3.2 eV 的绝缘体。**同样的骨架，完全不同的器件用途**。

### 练习 06a-2：掺杂浓度与超胞大小

**题目回顾**：构建不同大小的超胞，把一个 Sr 换成 La，考察可达到的最低掺杂浓度与 DFT 计算量。

In [ ]:
sto = structs["SrTiO3"]

rows = []
for n in (1, 2, 3, 4):
    sc = sto * (n, n, n)
    sr_idx = [i for i, s in enumerate(sc.sites) if s.specie.symbol == "Sr"]
    doped = sc.copy()
    doped.replace(sr_idx[0], "La")
    rows.append({"supercell": f"{n}x{n}x{n}", "n": n, "n_atoms": sc.num_sites,
                 "N_Sr": len(sr_idx), "x_min": round(1 / len(sr_idx), 5),
                 "doped_formula": doped.composition.reduced_formula,
                 "V_A3": round(sc.volume, 1)})

sup = pd.DataFrame(rows)
print(sup.to_string(index=False))

TARGET_X = 0.01
n_need = int(np.ceil((1 / TARGET_X) ** (1 / 3)))
sc_need = sto * (n_need, n_need, n_need)
print(f"\n要达到 x = {TARGET_X}: 至少需要 {n_need}x{n_need}x{n_need} 超胞")
print(f"  原子数 = {sc_need.num_sites}，Sr 位 = {n_need ** 3}，实际 x = {1 / n_need ** 3:.4f}")
print(f"  按 O(N^3) 估计，计算量约是原胞的 {(sc_need.num_sites / sto.num_sites) ** 3:.3g} 倍")

fig, ax = plt.subplots(figsize=(7.2, 4.3))
ax.plot(sup.n, sup.x_min, "o-", lw=1.8, ms=9, color=C.navy)
for _, r in sup.iterrows():
    ax.annotate(f"{r.n_atoms} atoms", (r.n, r.x_min),
                textcoords="offset points", xytext=(8, 6), fontsize=8)
ax.axhline(TARGET_X, color=C.crimson, ls="--", lw=1.3, label=f"target x = {TARGET_X}")
ax.set_yscale("log")
ax.set_xlabel("supercell multiple n")
ax.set_ylabel("lowest reachable doping level  x = 1 / N_Sr")
ax.set_title("Supercell size sets the accessible doping concentration")
ax.legend(fontsize=9)
plt.show()

**解析**：注意 $n = 1$ 那一行——原胞里只有 1 个 Sr，换掉它就得到 $\text{LaTiO}_3$，相当于 **100% 取代**，这根本不是“掺杂”。

关键的数量级：

| 目标浓度 | 需要的超胞 | 原子数 | 相对原胞的计算量 |
|---|---|---|---|
| 12.5% | $2\times2\times2$ | 40 | $\sim 5\times10^2$ |
| 3.7% | $3\times3\times3$ | 135 | $\sim 2\times10^4$ |
| 1.6% | $4\times4\times4$ | 320 | $\sim 3\times10^5$ |
| **1%** | $5\times5\times5$ | **625** | $\sim 2\times10^6$ |

这就是**稀磁半导体、稀土掺杂发光材料、低浓度缺陷**这类问题在第一性原理层面极其昂贵的原因——625 个原子的结构优化在普通集群上要跑几天到几周。

实践中的常见对策：
1. 用**特殊准随机结构（SQS）** 在较小的胞里模拟无序固溶体；
2. 用**机器学习势函数（MLIP）** 替代 DFT（第十讲的 MACE 正是干这个的）；
3. 接受较高的名义浓度，只提取随浓度变化的**趋势**而非绝对值。

### 练习 06a-3：用 COD 做一次真实的物相调研

**题目回顾**：检索 $\text{BaTiO}_3$ 的全部 COD 条目，统计空间群分布，对比四方相与立方相，并扫描 `symprec` 找出对称性被抹平的临界值。

In [ ]:
COD_SEARCH = "https://www.crystallography.net/cod/result"

def cod_search(timeout=40, **params):
    params["format"] = "json"
    try:
        r = requests.get(COD_SEARCH, params=params, timeout=timeout)
        return pd.DataFrame(r.json()) if r.status_code == 200 else pd.DataFrame()
    except Exception as exc:
        print("COD 检索失败（可能是网络不通）:", exc)
        return pd.DataFrame()

# 注意 COD 的化学式格式：元素按字母序、空格分隔、省略下标 1
hits = cod_search(formula="Ba O3 Ti")
if len(hits):
    print(f"COD 中 BaTiO3 共 {len(hits)} 条记录\n")
    print("空间群分布（前 5）：")
    print(hits["sg"].value_counts().head(5).to_string())
else:
    print("未取到 COD 数据，下面改用内置文献参数继续。")

In [ ]:
# 两个相：高温立方顺电相 与 室温四方铁电相（内置文献参数，离线也能跑）
cubic = Structure.from_spacegroup(
    "Pm-3m", Lattice.cubic(4.006), ["Ba", "Ti", "O"],
    [[0, 0, 0], [0.5, 0.5, 0.5], [0.5, 0.5, 0.0]])

tetra = Structure.from_spacegroup(
    "P4mm", Lattice.tetragonal(3.994, 4.038), ["Ba", "Ti", "O", "O"],
    [[0, 0, 0], [0.5, 0.5, 0.515], [0.5, 0.5, -0.025], [0.5, 0.0, 0.485]])

for label, s in [("cubic (high-T)", cubic), ("tetragonal (RT)", tetra)]:
    lat = s.lattice
    sga = SpacegroupAnalyzer(s, symprec=0.01)
    print(f"{label:16s} {s.composition.reduced_formula}  "
          f"a={lat.a:.3f}  c={lat.c:.3f}  c/a-1={lat.c / lat.a - 1:+.4f}  "
          f"rho={s.density:.3f}  SG={sga.get_space_group_symbol()}")

print("\n--- symprec 扫描：四方畸变什么时候被抹平？---")
scan = []
for sp in (0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.5):
    sga = SpacegroupAnalyzer(tetra, symprec=sp)
    scan.append({"symprec": sp, "space_group": sga.get_space_group_symbol(),
                 "SG_no": sga.get_space_group_number(),
                 "n_ops": len(sga.get_symmetry_operations())})
print(pd.DataFrame(scan).to_string(index=False))

print(f"\n参考：c - a = {tetra.lattice.c - tetra.lattice.a:.4f} A")
print(f"      Ti 沿 c 的位移 ~ {0.015 * tetra.lattice.c:.4f} A")
print(f"      O  沿 c 的位移 ~ {0.025 * tetra.lattice.c:.4f} A")

**解析**

**空间群分布**正好对应 $\text{BaTiO}_3$ 著名的四个相。它随温度依次经历：

$$\text{菱方 } R3m \;\longrightarrow\; \text{正交 } Amm2 \;\longrightarrow\; \text{四方 } P4mm \;\longrightarrow\; \text{立方 } Pm\bar{3}m$$

COD 的检索结果里这四个空间群都占了可观比例——**数据库的统计分布直接反映了物理相图**。前三个相都是铁电相（没有对称中心，允许自发极化），只有高温立方相是顺电的。

**$c/a - 1 \approx 0.011$**：四方畸变只有百分之一量级。正是这个微小畸变——Ti 相对氧八面体中心偏移约 0.06 Å——产生了 $\text{BaTiO}_3$ 的自发极化，使它成为最重要的铁电/压电材料之一。

**symprec 扫描**给出了最实际的教训：

- `symprec` 从 0.01 一路到 0.2，都正确识别出 $P4mm$（8 个对称操作）；
- 一旦超过约 **0.25**，spglib 就认为那点畸变在容差范围内可以忽略，于是判成立方 $Pm\bar{3}m$（48 个操作）。

临界值落在 0.2–0.3 之间并不意外：Ti 的位移约 0.06 Å，O 的位移约 0.10 Å，而 $c - a = 0.044$ Å。当容差超过这些位移的量级，结构就“看起来”是立方的了。

> **📌 一句话**：`symprec` 设得太大，你会把一个铁电体报告成顺电体。**报告空间群时必须同时报告 `symprec`**——这不是迂腐，是结论成立的前提。

---

# `06b` — XRD、凸包与筛选练习答案

### 练习 06b-1：晶格常数变化如何移动衍射峰

**题目回顾**：让金红石的晶格等比例膨胀 0–3%，记录 $(110)$ 峰位移，并与 $\Delta(2\theta) \approx -2\varepsilon\tan\theta$ 对比。

In [ ]:
EPS = [0.0, 0.01, 0.02, 0.03]
A0, C0 = 4.5937, 2.9587

def rutile(eps):
    return Structure.from_spacegroup(
        "P4_2/mnm", Lattice.tetragonal(A0 * (1 + eps), C0 * (1 + eps)),
        ["Ti", "O"], [[0, 0, 0], [0.3053, 0.3053, 0]])

rows = []
for eps in EPS:
    p = xrd.get_pattern(rutile(eps), two_theta_range=(20, 60))
    k = int(np.argmax(p.y))
    rows.append({"eps": eps, "a_A": round(A0 * (1 + eps), 4),
                 "hkl": "".join(str(abs(v)) for v in p.hkls[k][0]["hkl"]),
                 "two_theta": round(float(p.x[k]), 4),
                 "d_A": round(float(p.d_hkls[k]), 4)})

shift = pd.DataFrame(rows)
shift["measured_shift"] = (shift.two_theta - shift.two_theta.iloc[0]).round(4)

# 理论预测：d -> (1+eps)d  =>  Delta(2theta) ≈ -2 * eps * tan(theta)
theta0 = np.radians(shift.two_theta.iloc[0] / 2)
shift["predicted_shift"] = (-2 * shift.eps * np.tan(theta0) * 180 / np.pi).round(4)

print(shift.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.3))

ax = axes[0]
for eps in EPS:
    p = xrd.get_pattern(rutile(eps), two_theta_range=(26, 29))
    ax.vlines(p.x, 0, p.y, lw=2.2, label=f"eps = {eps:.0%}")
ax.set_xlabel("2theta / degree")
ax.set_ylabel("I (rel.)")
ax.set_title("(110) peak shifts to lower angle as the cell expands", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(shift.eps * 100, shift.measured_shift, "o-", lw=1.8, ms=9,
        color=C.navy, label="pymatgen 实测")
ax.plot(shift.eps * 100, shift.predicted_shift, "s--", lw=1.5, ms=7,
        color=C.crimson, label="理论 -2*eps*tan(theta)")
ax.set_xlabel("lattice expansion eps / %")
ax.set_ylabel("peak shift  d(2theta) / degree")
ax.set_title("Measured vs predicted peak shift", fontsize=10)
ax.legend(fontsize=9)
plt.show()

p_ref = xrd.get_pattern(rutile(0.0), two_theta_range=(20, 120))
print("同样 1% 膨胀下，不同角度峰的位移（灵敏度正比于 tan(theta)）：")
for tt in [float(v) for v in p_ref.x[:8]]:
    th = np.radians(tt / 2)
    print(f"  2theta = {tt:7.2f} deg  ->  d(2theta) = "
          f"{-2 * 0.01 * np.tan(th) * 180 / np.pi:+7.4f} deg")

**解析**

**推导**：对布拉格定律 $2d\sin\theta = \lambda$ 两边取对数再微分（$\lambda$ 固定）：

$$\frac{\Delta d}{d} + \cot\theta\,\Delta\theta = 0 \quad\Longrightarrow\quad \Delta\theta = -\varepsilon\tan\theta$$

其中 $\varepsilon = \Delta d / d$。由于实验记录的是 $2\theta$，所以 $\Delta(2\theta) = -2\varepsilon\tan\theta$。

数值吻合得很好（1% 膨胀：实测 $-0.277^\circ$，预测 $-0.280^\circ$）。剩下的微小偏差来自线性化——$\tan\theta$ 本身随 $\theta$ 变化，膨胀量越大偏差越明显。

**回答第 4 问：高角度峰灵敏得多。** 位移正比于 $\tan\theta$，而 $\tan\theta$ 在 $\theta \to 90^\circ$ 时发散。最后那段输出很直观：$2\theta \approx 27^\circ$ 的峰只移动 $0.28^\circ$，而 $2\theta \approx 110^\circ$ 附近的峰能移动好几度。

> **📌 实验要点**：精修晶胞参数时**一定要用高角度峰**（通常取 $2\theta > 80^\circ$），这是 Nelson–Riley 外推法等经典方法的理论基础。反过来，只盯着低角度的最强峰去判断“掺杂有没有进晶格”，灵敏度往往不足以分辨。

### 练习 06b-2：另一个二元体系的凸包

**题目回顾**：把凸包流程封装成 `build_hull(system)`，对 Al–O、Li–O、Fe–O 三个体系作分析。

In [ ]:
ea = pd.read_csv(find_file("energy_per_atom_aflow.csv"))
TOKEN = re.compile(r"([A-Z][a-z]?)(\d*\.?\d*)")

def parse_formula(f):
    """把 'O2Ti1' 解析成 {'O': 2.0, 'Ti': 1.0}。"""
    d = {}
    for el, n in TOKEN.findall(f):
        if el:
            d[el] = d.get(el, 0.0) + (float(n) if n else 1.0)
    return d

ea["comp"] = ea.formula.map(parse_formula)
ea["n_el"] = ea.comp.map(len)

elemental = ea[ea.n_el == 1].copy()
elemental["element"] = elemental.comp.map(lambda c: next(iter(c)))

OUTLIER_TOL = 2.0
mu, flagged = {}, []
for el, g in elemental.groupby("element"):
    med = g.energy_per_atom.median()
    dev = (g.energy_per_atom - med).abs()
    ok, bad = g[dev <= OUTLIER_TOL], g[dev > OUTLIER_TOL]
    for f, v in zip(bad.formula, bad.energy_per_atom):
        flagged.append({"element": el, "formula": f,
                        "energy": round(v, 3), "median": round(med, 3)})
    mu[el] = ok.energy_per_atom.min() if len(ok) else med

def formation_energy(comp, e_per_atom):
    total = sum(comp.values())
    if not all(el in mu for el in comp):
        return np.nan
    return e_per_atom - sum(n / total * mu[el] for el, n in comp.items())

ea["E_f"] = [formation_energy(c, v) for c, v in zip(ea.comp, ea.energy_per_atom)]
print(f"{len(mu)} 个元素有参考态；异常条目 {len(flagged)} 条")

In [ ]:
def build_hull(system, ax=None, verbose=True):
    """对一个二元体系求下凸包，返回 (全部条目带 E_hull, 凸包顶点坐标)。"""
    sub = ea[ea.comp.map(lambda c: len(c) > 0 and set(c) <= system)].copy()
    # 横轴取其中一个元素的原子分数；含 O 的体系习惯上把 O 放右端
    ref = "O" if "O" in system else sorted(system)[1]
    sub["x"] = sub.comp.map(lambda c: c.get(ref, 0) / sum(c.values()))
    sub["label"] = sub.comp.map(lambda c: Composition(
        "".join(f"{k}{v:g}" for k, v in c.items())).reduced_formula)

    # 每个组成只保留能量最低的多晶型
    best = sub.loc[sub.groupby("x").E_f.idxmin()].reset_index(drop=True)
    pts = np.column_stack([best.x.values, best.E_f.values])
    aug = np.vstack([pts, [0.5, pts[:, 1].max() + 10.0]])
    idx = sorted(i for i in ConvexHull(aug).vertices if i < len(pts))
    hull_pts = pts[idx]
    sub["E_hull"] = [ef - np.interp(x, hull_pts[:, 0], hull_pts[:, 1])
                     for x, ef in zip(sub.x, sub.E_f)]

    name = "-".join(sorted(system))
    if verbose:
        print(f"[{name}] 共 {len(sub)} 条记录，凸包上的稳定相：")
        for i in idx:
            print(f"    x({ref}) = {best.x[i]:.4f}   {best.label[i]:10s} "
                  f"E_f = {best.E_f[i]:+.4f} eV/atom")

    if ax is not None:
        ax.scatter(sub.x, sub.E_f, s=42, color="#b0b0b0", zorder=2)
        ax.plot(hull_pts[:, 0], hull_pts[:, 1], "-o", color=C.crimson,
                lw=1.9, ms=8, zorder=4)
        for i in idx:
            ax.annotate(best.label[i], (best.x[i], best.E_f[i]),
                        textcoords="offset points", xytext=(0, -16),
                        ha="center", fontsize=8, color=C.crimson)
        ax.axhline(0, color="#888888", lw=0.9, ls="--")
        ax.set_xlabel(f"x({ref})")
        ax.set_ylabel("E_f / eV per atom")
        ax.set_title(name, fontsize=11)
    return sub, hull_pts


def closest_metastable(sub, min_hull=1e-3):
    """找出 E_hull 最小的非稳定化合物（排除单质），并给出分解路径。"""
    cand = sub[(sub.E_hull > min_hull) & (sub.comp.map(len) > 1)]
    if not len(cand):
        print("    （该体系没有高于凸包的化合物条目）")
        return
    r = cand.loc[cand.E_hull.idxmin()]
    entries = [ComputedEntry(Composition("".join(f"{k}{v:g}" for k, v in c.items())),
                             v * sum(c.values()))
               for c, v in zip(sub.comp, sub.energy_per_atom)]
    pdg = PhaseDiagram(entries)
    target = min([e for e in entries if e.composition.reduced_formula == r.label],
                 key=lambda e: e.energy_per_atom)
    decomp, ehull = pdg.get_decomp_and_e_above_hull(target)
    route = "  +  ".join(f"{amt:.2f} {d.composition.reduced_formula}"
                         for d, amt in decomp.items())
    print(f"    最接近凸包的亚稳化合物: {r.label}  "
          f"E_hull = {ehull * 1000:.1f} meV/atom  ->  {route}")


SYSTEMS = [{"Al", "O"}, {"Li", "O"}, {"Fe", "O"}]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
for ax, sysset in zip(axes, SYSTEMS):
    sub_sys, _ = build_hull(sysset, ax=ax)
    closest_metastable(sub_sys)
    print()
plt.show()

**解析**

**Al–O**：凸包上只有 $\text{Al}_2\text{O}_3$（刚玉），$E_f \approx -3.02$ eV/atom（实验 $-3.47$）。与常识完全一致——铝只有一个稳定氧化物。

**Li–O**：$\text{Li}_2\text{O}$（$\approx -1.84$）和过氧化锂 $\text{Li}_2\text{O}_2$（$\approx -1.42$）**都在凸包上**。这个结果有直接的工程意义：锂空气电池的放电产物就是 $\text{Li}_2\text{O}_2$，它既然在凸包上，就说明它是热力学稳定相，不会自发歧化成 $\text{Li}_2\text{O} + \text{O}_2$。电池的可逆性问题出在动力学（$\text{Li}_2\text{O}_2$ 绝缘、充电过电位高），不在热力学。

**Fe–O —— 这里出了问题**：凸包上只有 $\text{Fe}_3\text{O}_4$（磁铁矿）。但相图教科书告诉我们，Fe–O 体系有**三个**常见氧化物：方铁矿 $\text{Fe}_{1-x}\text{O}$、磁铁矿 $\text{Fe}_3\text{O}_4$、赤铁矿 $\text{Fe}_2\text{O}_3$。**赤铁矿没上凸包，这是个明确的失败。**

原因正是题目提示的那一条：**Fe 是强关联的 $3d$ 过渡金属**。

1. **自相互作用误差**：普通 GGA/PBE 泛函对局域 $3d$ 电子存在自相互作用误差，会系统性地偏估铁氧化物的形成能，而且误差随氧化态而变——$\text{Fe}^{3+}$ 与 $\text{Fe}^{2+}$ 的误差不同，于是相对稳定性的顺序被搞错；
2. 标准对策是 **DFT+$U$**：对 Fe 的 $3d$ 轨道加一个 Hubbard $U$（铁氧化物常用 $U \approx 4$ eV）。Materials Project 对过渡金属氧化物默认启用 +$U$，正是为此；
3. 还有**磁序**问题：$\text{Fe}_2\text{O}_3$ 是反铁磁的，$\text{Fe}_3\text{O}_4$ 是亚铁磁的。磁构型设置不当，能量能差出零点几 eV/atom——足以颠覆凸包。

> **📌 这道题真正的教训**：**凸包的可靠性完全取决于底层 DFT 的可靠性**。凸包方法本身没错，错的是喂给它的能量。看到一个与实验相图矛盾的凸包时，第一反应不该是“实验错了”，而应该是“我的泛函、磁构型、参考态有没有问题”。

### 练习 06b-3：设计你自己的筛选流程——透明导电氧化物（TCO）

**题目回顾**：实现 TCO 筛选漏斗，先做健全性检验，诊断 PBE 偏差，再用修正后的阈值重跑。

In [ ]:
bg = pd.read_csv(find_file("bandgap_aflow.csv"))
bg["comp"] = bg.formula.map(parse_formula)
bg["n_el"] = bg.comp.map(len)
bg["elements"] = bg.comp.map(lambda c: frozenset(c))

pool = bg.merge(ea[["formula", "E_f"]], on="formula", how="left")
pool["key"] = pool.formula.map(lambda f: Composition(f).reduced_composition)

TOXIC       = set("Pb Cd Hg As Tl Be Cr Ni Sb".split())
RADIOACTIVE = set("Tc Pm Po At Rn Fr Ra Ac Th Pa U Np Pu".split())
NOBLE       = set("He Ne Ar Kr Xe".split())
SCARCE      = set("Au Pt Ir Os Rh Ru Pd Re Sc Hf Ta".split())
BLOCKED     = TOXIC | RADIOACTIVE | NOBLE | SCARCE

def tco_funnel(gap_min, ef_cut=-1.0, verbose=True):
    stages = [{"stage": "起始候选库", "n": len(pool)}]
    s = pool
    def gate(df, label):
        stages.append({"stage": label, "n": len(df)})
        if verbose:
            print(f"  {label:28s} -> {len(df):6d}")
        return df
    if verbose:
        print(f"  {'起始候选库':28s} -> {len(pool):6d}")
    s = gate(s[s.elements.map(lambda e: "O" in e)], "① 必须含氧")
    s = gate(s[s.band_gap > gap_min], f"② 带隙 > {gap_min} eV")
    s = gate(s[s.n_el <= 3], "③ 元素数 <= 3")
    s = gate(s[s.E_f < ef_cut], f"④ Ef < {ef_cut} eV/atom")
    s = gate(s[~s.elements.map(lambda e: bool(e & BLOCKED))], "⑤ 元素安全可得")
    return s.reset_index(drop=True), pd.DataFrame(stages)

KNOWN_TCO = {"In2O3": 3.75, "SnO2": 3.60, "ZnO": 3.37, "Ga2O3": 4.80}

print("=== 第一次尝试：直接用实验上的透明阈值 3.0 eV ===")
tco_v1, funnel_v1 = tco_funnel(3.0)
keys_v1 = set(tco_v1.key)
print("\n  健全性检验:")
for m in KNOWN_TCO:
    print(f"    {m:8s} 通过筛选: {Composition(m).reduced_composition in keys_v1}")

In [ ]:
print("诊断：这四个已知 TCO 的 PBE 带隙到底是多少？\n")
print(f"{'material':10s}{'PBE':>9s}{'实验':>9s}{'PBE/实验':>11s}   卡在哪一关")
ratios = []
for m, eg_exp in KNOWN_TCO.items():
    k = Composition(m).reduced_composition
    r = pool[pool.key == k]
    if not len(r):
        print(f"{m:10s}{'不在候选库中':>26s}")
        continue
    row = r.iloc[0]
    ratios.append(row.band_gap / eg_exp)
    if row.band_gap <= 3.0:
        why = "关卡②（PBE 带隙不够宽）"
    elif row.n_el > 3:
        why = "关卡③"
    elif not (row.E_f < -1.0):
        why = f"关卡④ (Ef={row.E_f:+.2f})"
    elif row.elements & BLOCKED:
        why = "关卡⑤"
    else:
        why = "（通过）"
    print(f"{m:10s}{row.band_gap:9.3f}{eg_exp:9.2f}{row.band_gap / eg_exp:11.2f}   {why}")

mean_ratio = float(np.mean(ratios))
print(f"\n平均 PBE/实验 比值 = {mean_ratio:.2f}")
print(f"=> 实验阈值 3.0 eV 映射到 PBE 尺度约为 {3.0 * mean_ratio:.2f} eV")

In [ ]:
GAP_MIN_PBE = 1.0        # 由上面的比例估计，取一个略保守的整数

print(f"=== 第二次尝试：PBE 尺度阈值 {GAP_MIN_PBE} eV ===")
tco, funnel_v2 = tco_funnel(GAP_MIN_PBE)
keys_v2 = set(tco.key)

print("\n  健全性检验（修正后）：")
for m in KNOWN_TCO:
    k = Composition(m).reduced_composition
    r = pool[pool.key == k]
    ok = k in keys_v2
    note = ""
    if not ok and len(r):
        row = r.iloc[0]
        if row.band_gap <= GAP_MIN_PBE:
            note = f"仍卡在关卡②: PBE Eg={row.band_gap:.3f}"
        elif not (row.E_f < -1.0):
            note = f"卡在关卡④: Ef={row.E_f:+.2f}"
        elif row.n_el > 3:
            note = "卡在关卡③"
        else:
            note = "卡在关卡⑤"
    print(f"    {m:8s} {str(ok):6s} {note}")

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.7))

ax = axes[0]
w = 0.38
idx = np.arange(len(funnel_v2))
ax.barh(idx - w / 2, funnel_v1.n, height=w, color="#c7c7c7", label="阈值 3.0 eV（错误）")
ax.barh(idx + w / 2, funnel_v2.n, height=w, color=C.gold,
        label=f"阈值 {GAP_MIN_PBE} eV（PBE 修正）")
for i, (n1, n2) in enumerate(zip(funnel_v1.n, funnel_v2.n)):
    ax.text(n1 * 1.1, i - w / 2, f"{n1:,}", va="center", fontsize=7.5, color="#666666")
    ax.text(n2 * 1.1, i + w / 2, f"{n2:,}", va="center", fontsize=7.5)
ax.set_yticks(idx)
ax.set_yticklabels(funnel_v2.stage, fontsize=8.5)
ax.invert_yaxis()
ax.set_xscale("log")
ax.set_xlim(right=funnel_v2.n.max() * 5)
ax.set_xlabel("剩余候选数（对数坐标）")
ax.set_title("TCO 筛选漏斗", fontsize=11)
ax.legend(fontsize=8, loc="lower right")

ax = axes[1]
ax.scatter(pool.band_gap, pool.E_f, s=4, alpha=0.10, color="#b0b0b0",
           edgecolor="none", label=f"全库 ({len(pool):,})")
ax.scatter(tco.band_gap, tco.E_f, s=18, color=C.crimson,
           edgecolor="none", label=f"入围 ({len(tco)})")
for m in KNOWN_TCO:
    r = pool[pool.key == Composition(m).reduced_composition]
    if len(r):
        ax.scatter(r.band_gap, r.E_f, s=140, marker="*", color=C.navy, zorder=5)
        ax.annotate(m, (r.band_gap.iloc[0], r.E_f.iloc[0]),
                    textcoords="offset points", xytext=(7, 5), fontsize=8, color=C.navy)
ax.axvline(GAP_MIN_PBE, color=C.gold, ls="--", lw=1.4)
ax.axhline(-1.0, color=C.navy, ls="--", lw=1.1)
ax.set_xlim(0, 7)
ax.set_ylim(-4.5, 0.5)
ax.set_xlabel("PBE band gap / eV")
ax.set_ylabel("Formation energy / eV per atom")
ax.set_title("已知 TCO 在性质空间中的位置", fontsize=11)
ax.legend(fontsize=8.5, loc="lower right")
plt.show()

**解析**

和正文里的光伏案例是同一个剧本，只是这次更夸张：

| 材料 | PBE 带隙 | 实验带隙 | 比值 |
|---|---|---|---|
| $\text{In}_2\text{O}_3$ | $\approx 1.11$ | 3.75 | **0.30** |
| $\text{SnO}_2$ | $\approx 0.00$ | 3.60 | **0.00** |
| ZnO | $\approx 1.82$ | 3.37 | 0.54 |
| $\text{Ga}_2\text{O}_3$ | $\approx 2.65$ | 4.80 | 0.55 |

直接用 3.0 eV 作阈值，**四个已知 TCO 一个都筛不出来**。把阈值按比例降到 PBE 尺度（约 1.0 eV）后，$\text{In}_2\text{O}_3$、ZnO、$\text{Ga}_2\text{O}_3$ 都回来了。

**$\text{SnO}_2$ 是救不回来的特例**：PBE 直接算出 0 eV 带隙，把这个宽带隙透明氧化物判成了金属。这与正文里 $\text{CuInSe}_2$ 的情况同源——都是 PBE 在含满 $d$ 壳层阳离子的化合物上的系统性失败。

**回答第 5 问**：TCO 这边的 PBE 偏差**比光伏那边更严重**。光伏材料的比值在 0.2–0.5，TCO 在 0.0–0.55，而且绝对误差更大（$\text{In}_2\text{O}_3$ 差了 2.6 eV，CdTe 只差 0.8 eV）。原因是带隙越宽，PBE 的自相互作用误差占比通常越大；再加上 $\text{In}_2\text{O}_3$、$\text{SnO}_2$ 这类材料的能带边由满 $d$ 带显著贡献，PBE 对 $d$ 态位置的描述本来就差。

> **📌 带走的结论**：**筛选阈值必须和数据的计算方法匹配**。拿实验尺度的判据去卡 PBE 数据，不是“保守”，而是**系统性地筛掉正确答案**。如果一定要用实验尺度的阈值，就得先做带隙校正（线性拟合、机器学习校正），或者干脆换用 HSE / GW 级别的数据。